# Downloading everything from Financial Modeling Prep before the subscription ends

**Carlo Hofer.** Run this in Google Colab. It downloads the market data the research
project needs, saves it to your Google Drive, and then writes a report telling you what
you actually got.

## Read this first

After cancellation none of this can be downloaded again. So the job is not "run the
code" — it is **prove you got everything, before you cancel**.

Three things make this different from a normal download script:

1. **It never overwrites a file.** If a file already exists it leaves it alone. A script
   in this project once destroyed hand-checked work by overwriting a file.
2. **It can be interrupted.** Colab disconnects after a while. Run the cell again — it
   skips everything already downloaded. You lose at most one request.
3. **It saves the raw answer from the server, untouched.** Tidying data can be redone
   later. A download cannot.

## Before you start

- A Google account, for Colab and Drive.
- Your FMP API key.

Work top to bottom. **Do not use Runtime → Run all.** Section 2 is a gate: if it fails,
stop and read what it says.

---
## 1. Setup

Nothing to install — Colab already has it. This mounts your Google Drive so the download
survives a disconnect. Google will ask permission; click through it.

In [ ]:
import os, json, gzip, time, random, hashlib
from datetime import datetime, timezone
from pathlib import Path
import requests

from google.colab import drive, userdata
drive.mount('/content/drive')

# Everything lands in your Drive, so a disconnect costs nothing.
OUT        = Path('/content/drive/MyDrive/fmp_extraction_2026-09')
RAW        = OUT / 'raw'
VENDOR_DCF = OUT / 'vendor_dcf_DO_NOT_OPEN'
LEDGER     = OUT / 'manifest_entries.jsonl'

for d in (OUT, RAW, VENDOR_DCF):
    d.mkdir(parents=True, exist_ok=True)

# FMP's own valuations get downloaded but must not be looked at yet. Reading them
# before your own model is finished would quietly reshape your assumptions.
guard = VENDOR_DCF / 'DO_NOT_OPEN.md'
if not guard.exists():
    guard.write_text(
        "Do not open these files.\n\n"
        "FMP's own DCF valuations. A check on your valuation model, to be compared\n"
        "only AFTER your model is finished and its assumptions written down and dated.\n")

print('Saving everything to:', OUT)

### Your API key

The key goes in **Colab's Secrets panel**, never in a cell. Anything typed into a cell is
saved inside the notebook file and can end up on GitHub.

Click the **key icon (🔑)** in the left sidebar → **Add new secret** → name it exactly
`FMP_API_KEY` → paste the key → switch on **Notebook access**.

In [ ]:
try:
    API_KEY = userdata.get('FMP_API_KEY')
except Exception:
    API_KEY = None

assert API_KEY, (
    "No FMP_API_KEY found.\n"
    "Open the key icon in the left sidebar, add a secret named exactly FMP_API_KEY,\n"
    "paste your key, switch on Notebook access, then run this cell again.")

print('Key loaded, length', len(API_KEY), '- the key itself is never printed.')

---
## 2. GATE: is your subscription the plan you think it is?

**Do not skip this.** On 2 September 2026 the FMP connection used in the Claude chat
turned out to be on the **free tier**, not Ultimate. At that tier almost none of this
data exists, and one endpoint quietly returned only five years of history with no error
at all — which looks exactly like success.

This asks FMP for one thing from each group and tells you plainly what you have.

In [ ]:
BASE = 'https://financialmodelingprep.com/stable'

def try_once(path, **params):
    """One request. Returns (status_code, parsed json or None)."""
    params['apikey'] = API_KEY
    try:
        r = requests.get(f'{BASE}/{path}', params=params, timeout=30)
    except Exception:
        return 0, None
    if r.status_code != 200:
        return r.status_code, None
    try:
        return 200, r.json()
    except Exception:
        return 200, None

GATE = [
    ('daily prices',          'historical-price-eod/full',        dict(symbol='DUK'),    'BLOCKING'),
    ('historical market cap', 'historical-market-capitalization', dict(symbol='DUK'),    'BLOCKING'),
    ('company profile',       'profile',                          dict(symbol='DUK'),    'BLOCKING'),
    ('delisted companies',    'delisted-companies',               dict(),                'BLOCKING'),
    ('FX rates',              'historical-price-eod/full',        dict(symbol='EURUSD'), 'BLOCKING'),
    ('income statement',      'income-statement',                 dict(symbol='DUK', period='annual', limit=2), 'needed'),
    ('earnings transcripts',  'earning-call-transcript-dates',    dict(symbol='DUK'),    'needed'),
    ('ESG scores',            'esg-ratings',                      dict(symbol='DUK'),    'nice to have'),
    ('analyst estimates',     'analyst-estimates',                dict(symbol='DUK', period='annual', limit=2), 'nice to have'),
]

print(f"{'what':<24}{'result':<14}{'importance'}")
print('-' * 58)
blocked = []
for label, path, params, importance in GATE:
    status, data = try_once(path, **params)
    ok = status == 200 and data not in (None, [], {})
    verdict = 'OK' if ok else (f'DENIED {status}' if status else 'no answer')
    if not ok and importance == 'BLOCKING':
        blocked.append(label)
    print(f'{label:<24}{verdict:<14}{importance}')
    time.sleep(0.4)

print()
if blocked:
    print('STOP. These are blocking and your plan refused them:')
    for b in blocked:
        print('   -', b)
    print()
    print('Check your plan at financialmodelingprep.com while logged in.')
    print('If it is not Ultimate, do not continue. The data you need cannot be')
    print('downloaded at this tier, and cancelling would make that permanent.')
else:
    print('All blocking endpoints answered. Continue to the truncation check.')

### The default-window trap

FMP has a habit worth knowing about: **if you do not give it a date range, it gives you
roughly the last five years and says nothing.** No error, no warning. It looks exactly
like a complete answer.

Verified on 2 September 2026: asked without dates, `AAPL` returned 2021-09-03 onward.
Asked for `from=2010-01-04`, the same endpoint returned 2010 data happily. The history is
there — you just have to ask for it.

Every download cell below passes an explicit `from` and `to`. The check below confirms
the server honours them on your key before you rely on it 2,400 times.

In [ ]:
status, data = try_once('historical-price-eod/full', symbol='EURUSD',
                        **{'from': '1990-01-01', 'to': '2026-09-02'})

if status != 200 or not data:
    print('Could not check - endpoint returned', status)
else:
    rows = data if isinstance(data, list) else data.get('historical', [])
    dates = sorted(r['date'] for r in rows if isinstance(r, dict) and 'date' in r)
    print('rows returned :', len(rows))
    print('earliest date :', dates[0] if dates else 'none')
    print('latest date   :', dates[-1] if dates else 'none')
    print()
    if dates and dates[0] > '2011-01-01':
        print('WARNING: you asked for 1990 onward and the earliest row is', dates[0] + '.')
        print('The date range was ignored. Do not run the download until you know why -')
        print('every dated file would come back short, and a short pull cannot be redone.')
    else:
        print('Date range honoured: history goes back to', dates[0] + '. Continue.')

---
## 3. The list of companies

304 symbols: **302 companies plus two benchmarks**, SPY and the S&P 500 index `^GSPC`.

The list is read from the project's own published manifest on GitHub, so it is the same
list the existing panel was built from rather than a new one typed here. The repo is
public: this only reads it, needs no GitHub login, and pushes nothing.

`sorted()` matters. A script in this project once produced results that could not be
reproduced because it looped over a Python `set`, whose order changes every run.

In [ ]:
REPO_URL = 'https://github.com/ochofer/paper1-hazard-exposure-data.git'
if not Path('/content/repo').exists():
    os.system(f'git clone --depth 1 {REPO_URL} /content/repo')

manifest = json.loads(Path('/content/repo/data/raw/manifest.json').read_text())
ALL_SYMBOLS = sorted({e['symbol'] for e in manifest['price_panel']['per_symbol']})
BENCHMARKS  = sorted(manifest['price_panel']['benchmark_symbols'])
COMPANIES   = sorted(s for s in ALL_SYMBOLS if s not in BENCHMARKS)

# Nine currencies in the universe. GBp is pence, ILA is agorot - both one hundredth of
# the main unit. FMP quotes FX in the main unit, so we fetch GBPUSD and ILSUSD and
# convert later. Nothing is converted here, deliberately.
CURRENCIES = ['AUD','CHF','DKK','EUR','GBp','ILA','NOK','SEK','USD']
MAJOR = {'GBp': 'GBP', 'ILA': 'ILS'}
FX_PAIRS = sorted({f"{MAJOR.get(c, c)}{base}"
                   for c in CURRENCIES for base in ('USD', 'EUR')
                   if MAJOR.get(c, c) != base})

print('total symbols :', len(ALL_SYMBOLS))
print('companies     :', len(COMPANIES))
print('benchmarks    :', BENCHMARKS)
print('FX pairs      :', len(FX_PAIRS), FX_PAIRS)

assert len(ALL_SYMBOLS) == 304, f'expected 304, got {len(ALL_SYMBOLS)}'
assert len(COMPANIES) == 302, f'expected 302 companies, got {len(COMPANIES)}'

---
## 4. What gets downloaded

Each row is one kind of data. **Tier 1** is what the project fails without. Tier 2 is
work already committed. Tier 3 is cheap now and impossible later.

`applies` matters: SPY and ^GSPC are a fund and an index, so their income statement or
ESG score would be meaningless. They only get prices.

In [ ]:
ENDPOINTS = [
  # tier 1 - blocking
  dict(tier=1, name='price_eod_full',          path='historical-price-eod/full',               applies='all',       dated=True),
  dict(tier=1, name='price_eod_dividend_adj',  path='historical-price-eod/dividend-adjusted',  applies='all',       dated=True),
  dict(tier=1, name='price_eod_non_split_adj', path='historical-price-eod/non-split-adjusted', applies='all',       dated=True),
  dict(tier=1, name='dividends',               path='dividends',                               applies='companies', dated=False),
  dict(tier=1, name='splits',                  path='splits',                                  applies='companies', dated=False),
  dict(tier=1, name='historical_market_cap',   path='historical-market-capitalization',        applies='companies', dated=True),
  dict(tier=1, name='shares_float',            path='shares-float',                            applies='companies', dated=False),
  dict(tier=1, name='profile',                 path='profile',                                 applies='all',       dated=False),
  dict(tier=1, name='delisted_companies',      path='delisted-companies',                      applies='once',      dated=False),
  dict(tier=1, name='fx_eod',                  path='historical-price-eod/full',               applies='fx',        dated=True),
  # tier 2
  dict(tier=2, name='income_statement_annual', path='income-statement',        applies='companies', dated=False, extra=dict(period='annual',  limit=200)),
  dict(tier=2, name='income_statement_qtr',    path='income-statement',        applies='companies', dated=False, extra=dict(period='quarter', limit=400)),
  dict(tier=2, name='balance_sheet_annual',    path='balance-sheet-statement', applies='companies', dated=False, extra=dict(period='annual',  limit=200)),
  dict(tier=2, name='balance_sheet_qtr',       path='balance-sheet-statement', applies='companies', dated=False, extra=dict(period='quarter', limit=400)),
  dict(tier=2, name='cash_flow_annual',        path='cash-flow-statement',     applies='companies', dated=False, extra=dict(period='annual',  limit=200)),
  dict(tier=2, name='cash_flow_qtr',           path='cash-flow-statement',     applies='companies', dated=False, extra=dict(period='quarter', limit=400)),
  dict(tier=2, name='key_metrics_annual',      path='key-metrics',             applies='companies', dated=False, extra=dict(period='annual',  limit=200)),
  dict(tier=2, name='ratios_annual',           path='ratios',                  applies='companies', dated=False, extra=dict(period='annual',  limit=200)),
  # tier 3
  dict(tier=3, name='transcript_dates',        path='earning-call-transcript-dates', applies='companies', dated=False),
  dict(tier=3, name='analyst_estimates',       path='analyst-estimates',       applies='companies', dated=False, extra=dict(period='annual', limit=200)),
  dict(tier=3, name='price_target_consensus',  path='price-target-consensus',  applies='companies', dated=False),
  dict(tier=3, name='esg_ratings',             path='esg-ratings',             applies='companies', dated=False),
  dict(tier=3, name='institutional_positions', path='institutional-ownership/symbol-positions-summary', applies='companies', dated=False),
  # vendor DCF - downloaded, then sealed
  dict(tier=3, name='vendor_dcf',              path='discounted-cash-flow',         applies='companies', dated=False, sealed=True),
  dict(tier=3, name='vendor_dcf_levered',      path='levered-discounted-cash-flow', applies='companies', dated=False, sealed=True),
]

def targets_for(ep):
    return {'all': ALL_SYMBOLS, 'companies': COMPANIES,
            'fx': FX_PAIRS, 'once': [None]}[ep['applies']]

for t in (1, 2, 3):
    n = sum(len(targets_for(e)) for e in ENDPOINTS if e['tier'] == t)
    print(f'tier {t}: {n:>5} requests')

---
## 5. Look before you loop

One request per endpoint first, printing the field names and the date range. Read it
before moving on. You are looking for two things.

- A field name that isn't what the report later expects: `date`, `symbol`, `currency`.
- A date range shorter than you asked for.

**Which of these have actually been checked.** On 2 September 2026 these seven were
confirmed against the live service and returned exactly the fields below:

| endpoint | fields returned |
|---|---|
| `historical-price-eod/full` | `symbol date open high low close volume change changePercent vwap` |
| `historical-price-eod/light` | `symbol date price volume` |
| `historical-market-capitalization` | `symbol date marketCap` |
| `income-statement` | `date symbol reportedCurrency cik filingDate fiscalYear period revenue …` |
| `dividends` | `symbol date recordDate paymentDate declarationDate adjDividend dividend yield frequency` |
| `splits` | `symbol date numerator denominator splitType` |
| `profile` | `symbol companyName currency exchange industry sector country isin cik cusip ipoDate …` |
| `delisted-companies` | `symbol companyName exchange ipoDate delistedDate` |

The rest — transcripts, ESG, 13F, analyst estimates, DCF, index constituents, shares
float — **could not be checked**, because the connection available for testing was
restricted to a handful of demo symbols. Those field names come from FMP's documentation
and may be wrong. Read their probe output especially carefully.

The DCF endpoints print their field names and **not their values**, on purpose.

In [ ]:
def probe(ep):
    tgt = targets_for(ep)[0]
    params = dict(ep.get('extra', {}))
    if tgt:
        params['symbol'] = tgt
    if ep.get('dated'):
        params['from'] = '1990-01-01'
        params['to'] = datetime.now(timezone.utc).strftime('%Y-%m-%d')
    status, data = try_once(ep['path'], **params)

    if status != 200 or data in (None, [], {}):
        print(f"{ep['name']:<26} DENIED or EMPTY  (status {status})\n")
        return
    rows = data if isinstance(data, list) else [data]
    first = rows[0] if rows else {}
    keys = sorted(first.keys()) if isinstance(first, dict) else type(first).__name__
    dates = sorted(r['date'] for r in rows if isinstance(r, dict) and isinstance(r.get('date'), str))
    span = f'{dates[0]} .. {dates[-1]}' if dates else 'no dates'
    print(f"{ep['name']:<26} OK  {len(rows):>6} rows  {span}")
    print(f"    fields: {keys}")
    if ep.get('sealed'):
        print('    [SEALED] values deliberately not shown')
    else:
        print(f"    first row: {json.dumps(first)[:220]}")
    print()

for ep in ENDPOINTS:
    probe(ep)
    time.sleep(0.4)

---
## 6. The download

Settings first. **Start conservative.** Being rate-limited into a ban a week before
cancellation is the one mistake that cannot be undone.

Look up your plan's requests-per-minute limit and put *half* of it in
`CALLS_PER_MINUTE` for the first run.

In [ ]:
CALLS_PER_MINUTE = 120        # half your plan's documented limit, for the first run
MIN_GAP          = 60.0 / CALLS_PER_MINUTE
MAX_RETRIES      = 6
STOP_AFTER_429S  = 5          # give up rather than risk a ban

_last = [0.0]
_consecutive_429 = [0]

def fetch(path, params):
    """One request, with polite waiting and backoff. Returns (status, raw bytes)."""
    params = dict(params)
    params['apikey'] = API_KEY
    for attempt in range(MAX_RETRIES):
        gap = time.monotonic() - _last[0]
        if gap < MIN_GAP:
            time.sleep(MIN_GAP - gap)
        _last[0] = time.monotonic()

        try:
            r = requests.get(f'{BASE}/{path}', params=params, timeout=60)
        except Exception:
            time.sleep(5 * (2 ** attempt))
            continue

        if r.status_code == 429:
            _consecutive_429[0] += 1
            if _consecutive_429[0] >= STOP_AFTER_429S:
                raise SystemExit(
                    'Too many rate-limit responses in a row. Stopping on purpose.\n'
                    'Lower CALLS_PER_MINUTE and run again - nothing downloaded is lost.')
            wait = min(600, 5 * (2 ** attempt)) + random.uniform(0, 3)
            print(f'    rate limited, waiting {wait:.0f}s')
            time.sleep(wait)
            continue

        _consecutive_429[0] = 0
        if r.status_code >= 500:
            time.sleep(min(600, 5 * (2 ** attempt)))
            continue
        return r.status_code, r.content
    return 0, b''


def save_never_overwriting(target, body):
    """Never destroys an existing file. Returns (path, what_it_did)."""
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        beside = target.with_name(target.name.replace('.json.gz', f'__refetch_{stamp}.json.gz'))
        with gzip.open(beside, 'wb') as fh:
            fh.write(body)
        return beside, 'WROTE_BESIDE'
    with gzip.open(target, 'wb') as fh:
        fh.write(body)
    return target, 'WROTE_NEW'


def summarise(body):
    """Row count and date range, read from a COPY. The saved file is never changed."""
    out = {'rows': None, 'date_min': None, 'date_max': None}
    try:
        data = json.loads(body)
    except Exception:
        return out
    rows = data if isinstance(data, list) else [data]
    out['rows'] = len(rows)
    dates = sorted(r['date'] for r in rows if isinstance(r, dict) and isinstance(r.get('date'), str))
    if dates:
        out['date_min'], out['date_max'] = dates[0], dates[-1]
    return out

print('Ready.', CALLS_PER_MINUTE, 'requests per minute.')

### Run it

Tier 1 first. It prints progress as it goes.

**If Colab disconnects, just run this cell again.** It checks what is already in your
Drive and skips it.

In [ ]:
TIERS_TO_RUN = [1]          # change to [2], then [3], once tier 1 is done

counts = {'WROTE_NEW': 0, 'WROTE_BESIDE': 0, 'SKIPPED': 0, 'EMPTY': 0, 'FAILED': 0}
ledger = LEDGER.open('a')

for ep in [e for e in ENDPOINTS if e['tier'] in TIERS_TO_RUN]:
    tgts = targets_for(ep)
    root = (VENDOR_DCF if ep.get('sealed') else RAW) / ep['name']
    print(f"\n=== tier {ep['tier']}  {ep['name']}  ({len(tgts)} requests) ===")

    for i, sym in enumerate(tgts, 1):
        target = root / f"{(sym or 'all').replace('/', '_')}.json.gz"
        if target.exists() and target.stat().st_size > 0:
            counts['SKIPPED'] += 1
            continue

        params = dict(ep.get('extra', {}))
        if sym:
            params['symbol'] = sym
        if ep.get('dated'):
            params['from'] = '1990-01-01'
            params['to'] = datetime.now(timezone.utc).strftime('%Y-%m-%d')

        status, body = fetch(ep['path'], params)
        if status != 200 or not body:
            counts['FAILED'] += 1
            ledger.write(json.dumps({'endpoint': ep['name'], 'symbol': sym,
                                     'http_status': status, 'disposition': 'FAILED',
                                     'fetched_utc': datetime.now(timezone.utc).isoformat()}) + '\n')
            ledger.flush()
            continue

        if body.strip() in (b'[]', b'{}'):
            counts['EMPTY'] += 1
        path, how = save_never_overwriting(target, body)
        counts[how] += 1
        meta = summarise(body)

        ledger.write(json.dumps({
            'endpoint': ep['name'], 'symbol': sym, 'path': ep['path'],
            'params': {k: v for k, v in params.items() if k != 'apikey'},
            'fetched_utc': datetime.now(timezone.utc).isoformat(),
            'http_status': status, 'bytes': len(body),
            'sha256': hashlib.sha256(body).hexdigest(),
            'file': str(path.relative_to(OUT)), 'disposition': how,
            'sealed': bool(ep.get('sealed')), **meta}) + '\n')
        ledger.flush()

        if i % 25 == 0 or i == len(tgts):
            print(f"  [{i}/{len(tgts)}] {sym}  rows={meta['rows']}  {meta['date_min']}..{meta['date_max']}")

ledger.close()
print('\n' + json.dumps(counts, indent=2))

---
## 7. The report — this is the actual deliverable

Downloading is not the goal. **Knowing what you got is the goal.** Run this after each
tier, and read it before you cancel anything.

It never opens the sealed DCF files; it only counts them.

In [ ]:
from collections import defaultdict

rows = [json.loads(l) for l in LEDGER.read_text().splitlines() if l.strip()]
good = {(r['endpoint'], r.get('symbol')): r for r in rows
        if str(r.get('disposition', '')).startswith('WROTE')}
by_ep = defaultdict(dict)
for (ep, sym), r in good.items():
    by_ep[ep][sym] = r

L = [f'# Completeness report - {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")} UTC', '']

L += ['## Coverage', '', '| data | got | empty | missing | of |', '|---|---:|---:|---:|---:|']
missing_notes = []
for ep in ENDPOINTS:
    if ep['applies'] == 'once':
        continue
    tgts, got = targets_for(ep), by_ep.get(ep['name'], {})
    usable  = [s for s in tgts if got.get(s) and (got[s].get('rows') or 0) > 0]
    empty   = [s for s in tgts if got.get(s) and not (got[s].get('rows') or 0)]
    missing = [s for s in tgts if s not in got]
    L.append(f"| {ep['name']} | {len(usable)} | {len(empty)} | {len(missing)} | {len(tgts)} |")
    if missing or empty:
        missing_notes.append((ep['name'], missing, empty))
L.append('')
for name, missing, empty in missing_notes:
    L.append(f'**{name}**')
    if missing:
        L.append(f'- never fetched ({len(missing)}): {", ".join(missing)}')
    if empty:
        L.append(f'- fetched but empty ({len(empty)}): {", ".join(empty)}')
    L.append('')

L += ['## Date ranges actually obtained', '']
for name in ('price_eod_full', 'price_eod_dividend_adj', 'historical_market_cap', 'fx_eod'):
    got = by_ep.get(name, {})
    if not got:
        L.append(f'- **{name}**: nothing downloaded.')
        continue
    mins = sorted(r['date_min'] for r in got.values() if r.get('date_min'))
    if not mins:
        L.append(f'- **{name}**: no dated rows.')
        continue
    late = sorted((r['date_min'], s) for s, r in got.items()
                  if r.get('date_min') and r['date_min'] > '2010-01-01')
    years = defaultdict(int)
    for r in got.values():
        if r.get('date_min'):
            years[r['date_min'][:4]] += 1
    top = sorted(years.items(), key=lambda kv: -kv[1])[:3]
    L.append(f'- **{name}**: earliest {mins[0]}; {len(late)} of {len(got)} start after 2010-01-01.')
    L.append(f'  Start-year clustering: {", ".join(f"{y} ({n})" for y, n in top)}. '
             'One dominant year means a plan cap, not real history.')
    if late:
        L.append('  Latest starters: ' + ', '.join(f'{s} ({d})' for d, s in sorted(late, reverse=True)[:15]))
L.append('')

dl = by_ep.get('delisted_companies', {}).get(None)
L += ['## Delisted companies', '']
if not dl:
    L.append('Not downloaded.')
else:
    recs = json.loads(gzip.open(OUT / dl['file'], 'rb').read())
    ds = sorted(r['delistedDate'] for r in recs
                if isinstance(r, dict) and r.get('delistedDate'))
    pre = sum(1 for d in ds if d < '2016')
    L.append(f'- {len(recs)} records; earliest {ds[0] if ds else "n/a"}, latest {ds[-1] if ds else "n/a"}')
    L.append(f'- before 2016: {pre} ({100 * pre / max(1, len(ds)):.1f}%) - measured, not assumed')
L.append('')

L += ['## Currencies', '']
seen = defaultdict(list)
for sym, r in sorted(by_ep.get('profile', {}).items()):
    try:
        rec = json.loads(gzip.open(OUT / r['file'], 'rb').read())
        rec = rec[0] if isinstance(rec, list) and rec else rec
        seen[str(rec.get('currency'))].append(sym)
    except Exception:
        seen['<unreadable>'].append(sym)
for c in sorted(seen):
    L.append(f'- {c}: {len(seen[c])} symbols')
absent = [c for c in CURRENCIES if c not in seen]
L.append(f'- expected nine: {", ".join(CURRENCIES)}')
L.append(f'- **absent: {", ".join(absent) if absent else "none"}**')
L.append('')
L.append(f'GBp is pence and ILA is agorot, both one hundredth of the main unit. '
         f'{len(seen.get("GBp", []))} symbols are quoted in GBp. No conversion has been '
         f'applied to the downloaded files. Divide by 100 before any value-weighting.')
L.append('')

def dirsize(p):
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())

def human(n):
    for u in ('B', 'KB', 'MB', 'GB'):
        if n < 1024:
            return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} TB'

sealed_n = len(list(VENDOR_DCF.rglob('*.json.gz')))
L += ['## Size and location', '',
      f'- working data: `{RAW}` - {human(dirsize(RAW))}',
      f'- sealed DCF: `{VENDOR_DCF}` - {human(dirsize(VENDOR_DCF))}, {sealed_n} files, **not opened**',
      f'- ledger: {len(rows)} entries', '']

fails = defaultdict(list)
for r in rows:
    if r.get('disposition') == 'FAILED':
        fails[r['endpoint']].append(r.get('http_status'))
L += ['## What your plan refused', '']
if not fails:
    L.append('Nothing was refused.')
else:
    for ep, v in sorted(fails.items()):
        L.append(f'- `{ep}`: {len(v)} failures, e.g. HTTP {v[0]}')
L.append('')

beside = [r for r in rows if r.get('disposition') == 'WROTE_BESIDE']
L += ['## Safety checks', '',
      f'- files written beside an existing file rather than over it: **{len(beside)}**']
if beside:
    L += [f"  - {r['endpoint']} / {r.get('symbol')} -> {r['file']}" for r in beside]
else:
    L.append('  - none; nothing was overwritten')
L += ['- symbols were sorted before looping; no Python set was iterated',
      '- every server response was saved exactly as received, before any parsing', '']

report = OUT / f'COMPLETENESS_REPORT_{datetime.now(timezone.utc).strftime("%Y-%m-%d")}.md'
if report.exists():
    report = report.with_name(report.stem + datetime.now(timezone.utc).strftime('__%H%M%SZ') + '.md')
report.write_text('\n'.join(L))
print('\n'.join(L))
print('\n\nSaved to:', report)

---
## 8. When it is finished

1. Read the report above. Every "missing" and every short date range is a question to
   answer **while the subscription is still live**.
2. Copy the folder `fmp_extraction_2026-09` out of Google Drive onto your own computer,
   so there are two copies.
3. Only then cancel.

### About GitHub

This notebook lives in the repo and you opened it from there, so pushing a change to it is
normal and fine — it is code, and the repo already publishes code.

**But do not use Colab's "Save a copy in GitHub" after running it.** A run fills the cells
with output, and that output contains FMP prices and company data. Saving that back to the
repo would push licensed data into a public repository — which is the one thing
`data/raw/` being gitignored exists to prevent.

The safe habit: edit and push from your machine, use Colab only to run. If you do change
something in Colab and want to keep it, choose **Edit → Clear all outputs** first, then
download the `.ipynb` and commit that.

The downloaded data itself never goes to GitHub at all. It lives in your Drive and on
your machine.

**Do not open `vendor_dcf_DO_NOT_OPEN`.** Those are FMP's own valuations. They are a
check on your model *after* your assumptions are written down and dated. Looking first
would quietly reshape your assumptions to match theirs, which is the thing this whole
project is built to avoid.